<a href="https://colab.research.google.com/github/JosephBless/Deep-Learning-Colab/blob/main/Chap_8%EF%BC%9A%E9%A2%A8%E9%9A%AA%E7%AE%A1%E7%90%86%E5%9F%BA%E7%A4%8E%E8%88%87%E9%A2%A8%E9%9A%AA%E8%AA%BF%E6%95%B4%E5%BE%8C%E7%9A%84%E6%94%B6%E7%9B%8A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install backtrader yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.9 MB/s eta 0:00:00


今天，我們將深入學習風險管理的基本概念，並探討如何利用風險調整後的收益指標來評估和優化投資策略的表現。風險管理在投資中至關重要，結合風險調整後的收益分析，能夠更全面地衡量策略的績效。

### **一、風險管理基礎**

#### **1. 風險管理的基本概念**

風險管理的核心概念是降低損失風險，確保投資組合在市場波動中能夠保持穩定。有效的風險管理可以保護投資資本，並在市場條件不利時減少潛在的損失。成熟的投資者皆會有自己的一套止損止盈的邏輯來讓自己可以穩定獲利以及避開過度跌幅！

#### **2. 常見的風險控制策略**

- **止損（Stop Loss）**：當資產價格下跌到設定的止損點時，系統會自動賣出，以防止更大的損失。止損是一種簡單但有效的風險控制策略。

  例如，如果我們設定止損點為10%，當資產價格從購買價格下跌10%時，則會自動賣出。
  ![https://ithelp.ithome.com.tw/upload/images/20240923/2012054990lWVXWmjt.jpg](https://ithelp.ithome.com.tw/upload/images/20240923/2012054990lWVXWmjt.jpg)
  
- **止盈（Take Profit）**：與止損相反，止盈是在資產價格達到設定的收益目標時，自動賣出以鎖定利潤，防止回吐收益。
![https://ithelp.ithome.com.tw/upload/images/20240923/20120549tXPlDnPtOK.jpg](https://ithelp.ithome.com.tw/upload/images/20240923/20120549tXPlDnPtOK.jpg)

- **倉位管理（Position Sizing）**：根據資產的風險水平和總資金量，決定每筆交易的資金投入，避免單一交易對投資組合造成重大影響。
![https://ithelp.ithome.com.tw/upload/images/20240923/20120549rS9fSWlDqY.jpg](https://ithelp.ithome.com.tw/upload/images/20240923/20120549rS9fSWlDqY.jpg)


### **二、風險調整後的收益**

風險調整後的收益能夠更準確地反映策略的績效，因為它考慮了獲利相對於風險的水平。以下是一些常用的風險調整後收益指標：

#### **1. Sortino比率（Sortino Ratio）**

Sortino比率是對夏普比率的改進，它只考慮負向波動（下行風險），對評估策略在承受損失方面的表現更為準確：
$$
Sortino\ Ratio = \frac{R_p - R_f}{\sigma_d}
$$
- $ R_p $：投資組合的平均收益率
- $ R_f $：無風險利率
- $ \sigma_d $：下行波動率

#### **2. 卡瑪比率（Calmar Ratio）**

卡瑪比率考慮最大回撤，衡量單位最大損失所獲得的收益：
$$
Calmar\ Ratio = \frac{R_p}{Max\ Drawdown}
$$
- $ R_p $：投資組合的年化收益率
- $ Max\ Drawdown $：最大回撤

#### **3. 夏普比率（Sharpe Ratio）**

夏普比率則衡量單位風險所獲得的超額收益，用來評估策略的風險調整後的收益。詳細可參考我們之前的介紹。

### **三、實作風險管理與風險調整後的收益計算**

以下示範如何運用 `Backtrader` 來設置風險管理策略並計算風險調整後的收益指標。

#### **1. 建立策略類別並計算風險調整後的收益**

In [ ]:
import backtrader as bt
import pandas as pd

class RiskAdjustedStrategy(bt.Strategy):
    params = (('period', 10), ('stop_loss', 0.1), ('take_profit', 0.2))

    def __init__(self):
        self.momentum = bt.indicators.Momentum(self.data.close, period=self.params.period)
        self.daily_values = []  # 用於儲存每日資產價值
        self.buy_price = None

    def next(self):
        if not self.position:  # 若無持倉
            if self.momentum[0] > 0:
                self.buy()
                self.buy_price = self.data.close[0]  # 紀錄買入價格
        else:
            # 實施止損和止盈策略
            if self.data.close[0] <= self.buy_price * (1 - self.params.stop_loss):
                self.sell()  # 觸發止損
            elif self.data.close[0] >= self.buy_price * (1 + self.params.take_profit):
                self.sell()  # 觸發止盈

        self.daily_values.append(self.broker.getvalue())

    def stop(self):
        self.returns = pd.Series(self.daily_values).pct_change().dropna()
        annual_return = self.returns.mean() * 252
        annual_volatility = self.returns.std() * np.sqrt(252)
        downside_risk = self.returns[self.returns < 0].std() * np.sqrt(252)
        max_drawdown = (self.returns / self.returns.cummax() - 1).min()

        self.sharpe_ratio = (annual_return - 0.01) / annual_volatility
        self.sortino_ratio = (annual_return - 0.01) / downside_risk if downside_risk != 0 else None
        self.calmar_ratio = annual_return / abs(max_drawdown) if max_drawdown != 0 else None

        print(f"夏普比率: {self.sharpe_ratio:.2f}, Sortino比率: {self.sortino_ratio:.2f}, 卡瑪比率: {self.calmar_ratio:.2f}")

#### **2. 設定並執行回測**

In [ ]:
cerebro = bt.Cerebro()
data = yf.download('AAPL', start='2020-01-01', end='2021-01-01')
data = bt.feeds.PandasData(dataname=data)
cerebro.adddata(data)
print("\n")

cerebro.addstrategy(RiskAdjustedStrategy, stop_loss=0.1, take_profit=0.2)
cerebro.broker.setcash(100000.0)
cerebro.run()

[*********************100%***********************]  1 of 1 completed



夏普比率: -25.29, Sortino比率: -33.35, 卡瑪比率: 0.00


### **四、策略優化與風險調整**

我們將通過調整策略中的關鍵參數，如止損、止盈、和動量週期等，以找出最能提高策略在風險調整後收益表現的參數組合。這一過程可以幫助我們找到在最大化收益的同時，最有效地控制風險的方法。

#### **1. 使用 `optstrategy` 進行參數優化**

`Backtrader` 提供了強大的 `optstrategy` 方法，使我們可以自動化地進行多個參數組合的測試，並找出表現最佳的參數組合。

#### **(1) 設定多個參數範圍**

假設我們想要優化以下幾個參數：

- **止損（stop_loss）**：0.05（5%）、0.1（10%）、0.15（15%）
- **止盈（take_profit）**：0.1（10%）、0.2（20%）、0.3（30%）
- **動量週期（period）**：從 5 到 20，每次增量為 5

In [ ]:
# 初始化 Cerebro 回測環境
cerebro = bt.Cerebro()

# 添加數據
data = yf.download('AAPL', start='2020-01-01', end='2021-01-01')
data = bt.feeds.PandasData(dataname=data)
cerebro.adddata(data)
print("\n")

# 使用 optstrategy 方法添加策略並指定參數範圍
cerebro.optstrategy(
    RiskAdjustedStrategy,
    stop_loss=[0.05, 0.1, 0.15],
    take_profit=[0.1, 0.2, 0.3],
    period=range(5, 21, 5)
)

# 設定初始資金和其他分析工具
cerebro.broker.setcash(100000.0)
cerebro.addsizer(bt.sizers.PercentSizer, percents=10)
cerebro.addanalyzer(btanalyzers.SharpeRatio, _name="sharpe", timeframe=bt.TimeFrame.Days, annualize=True)
cerebro.addanalyzer(btanalyzers.DrawDown, _name="drawdown")
cerebro.addanalyzer(btanalyzers.Returns, _name="returns")

# 執行策略優化
back = cerebro.run(maxcpus=1)

[*********************100%***********************]  1 of 1 completed




夏普比率: 1.08, Sortino比率: 1.45, 卡瑪比率: 0.02
夏普比率: 1.59, Sortino比率: 1.96, 卡瑪比率: 0.01
夏普比率: 1.76, Sortino比率: 2.33, 卡瑪比率: 0.03
夏普比率: 1.66, Sortino比率: 2.15, 卡瑪比率: 0.03
夏普比率: 1.53, Sortino比率: 2.16, 卡瑪比率: 0.02
夏普比率: 2.63, Sortino比率: 3.94, 卡瑪比率: 0.02
夏普比率: 2.35, Sortino比率: 3.37, 卡瑪比率: 0.03
夏普比率: 2.45, Sortino比率: 3.54, 卡瑪比率: 0.04
夏普比率: 0.93, Sortino比率: 1.27, 卡瑪比率: 0.02
夏普比率: 1.93, Sortino比率: 2.71, 卡瑪比率: 0.02
夏普比率: 1.63, Sortino比率: 2.22, 卡瑪比率: 0.03
夏普比率: 1.87, Sortino比率: 2.60, 卡瑪比率: 0.03
夏普比率: 1.31, Sortino比率: 1.77, 卡瑪比率: 0.02
夏普比率: 1.69, Sortino比率: 2.19, 卡瑪比率: 0.01
夏普比率: 1.41, Sortino比率: 1.86, 卡瑪比率: 0.02
夏普比率: 1.43, Sortino比率: 1.87, 卡瑪比率: 0.02
夏普比率: 1.07, Sortino比率: 1.47, 卡瑪比率: 0.02
夏普比率: 1.39, Sortino比率: 1.93, 卡瑪比率: 0.01
夏普比率: 1.71, Sortino比率: 2.35, 卡瑪比率: 0.02
夏普比率: 1.38, Sortino比率: 1.74, 卡瑪比率: 0.02
夏普比率: 1.17, Sortino比率: 1.64, 卡瑪比率: 0.02
夏普比率: 1.50, Sortino比率: 2.12, 卡瑪比率: 0.01
夏普比率: 1.39, Sortino比率: 1.88, 卡瑪比率: 0.02
夏普比率: 1.54, Sortino比率: 2.09, 卡瑪比率: 0.02
夏普比率: 1.48, Sortino比率: 2.03, 卡瑪比率: 0.0

這樣，`Backtrader` 將自動針對我們指定的所有參數組合進行回測，並且對每個組合計算夏普比率、Sortino比率和最大回撤等指標。

#### **(2) 分析優化結果**

完成優化後，我們需要對回測結果進行解析，找出最佳的參數組合。

In [ ]:
# 提取每個組合的回測結果
par_list = []
for run in back:
    for strategy in run:
        sharpe_ratio = strategy.analyzers.sharpe.get_analysis().get('sharperatio')
        returns = strategy.analyzers.returns.get_analysis().get('rnorm100')
        drawdown = strategy.analyzers.drawdown.get_analysis().get('max', {}).get('drawdown')

        if sharpe_ratio is not None:
            par_list.append([
                strategy.params.stop_loss,
                strategy.params.take_profit,
                strategy.params.period,
                returns,
                drawdown,
                sharpe_ratio
            ])

# 將結果轉換為 DataFrame 進行分析
par_df = pd.DataFrame(par_list, columns=['stop_loss', 'take_profit', 'period', 'return', 'drawdown', 'sharpe'])

# 找到夏普比率最高的參數組合
best_result = par_df.sort_values(by='sharpe', ascending=False).iloc[0]
print(f'最佳參數組合: 止損={best_result["stop_loss"]}, 止盈={best_result["take_profit"]}, 動量週期={best_result["period"]}')
print(f'對應的夏普比率: {best_result["sharpe"]:.2f}')

最佳參數組合: 止損=0.05, 止盈=0.2, 動量週期=10.0
對應的夏普比率: 2.56


#### **(3) 進一步的優化建議**

1. **引入更多參數**：可以將策略中的其他參數納入優化範圍，例如移動平均線的長度或 RSI 指標的超買超賣區間，這樣能夠更全面地優化策略。

2. **加強風險管理**：在優化過程中，除了關注夏普比率，也應該注意最大回撤（Max Drawdown）等風險指標，確保策略在收益與風險之間取得平衡。

3. **多資產測試**：嘗試在多個資產（如股票、ETF、外匯等）上運行優化策略，檢查參數組合是否對不同資產類別都適用。

4. **引入其他風險調整後收益指標**：除了夏普比率外，還可以使用 Sortino 比率和 Calmar 比率等進行綜合評估，以獲得更全面的優化結果。

---

### **五、優化結果的解讀與應用**

透過 `optstrategy` 的優化，我們能夠自動化地找到在風險調整後表現最佳的參數組合。這種方法不僅提高了策略的勝率，也確保策略在不同市場環境中都能保持穩定的績效。通過結合風險管理和優化技術，我們可以更有效地構建一個既能獲得高收益又能降低風險的投資組合。

總結來說，策略優化與風險調整的過程，能夠幫助投資者在實踐中找到最適合自己的投資策略，並在真實市場中提高策略的應用效率與成功率。

---

### **六、總結與作業**

今天的學習涵蓋了風險管理的基本概念與應用、以及如何利用風險調整後的收益指標來評估和優化策略。

**今日作業：**
1. 實施自己的策略，並計算其**夏普比率**、**Sortino比率**和**卡瑪比率**，分析策略的風險調整後收益。
2. 使用 `Backtrader` 進行策略優化，嘗試調整不同的風險管理參數，找出最佳的策略組合。
3. 比較不同風險管理策略對於投資組合表現的影響，找出最適合的風險管理方案。

透過這些實踐，你將能夠更熟練地管理風險，並運用風險調整後的收益來優化投資策略，在實際投資環境中做出更明智的決策。

## Appendix.優化完整程式如下:

In [ ]:
import backtrader as bt
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import backtrader.analyzers as btanalyzers

class RiskAdjustedStrategy(bt.Strategy):
    params = (('period', 10), ('stop_loss', 0.1), ('take_profit', 0.2))

    def __init__(self):
        self.momentum = bt.indicators.Momentum(self.data.close, period=self.params.period)
        self.daily_values = []  # 用於儲存每日資產價值
        self.buy_price = None

    def next(self):
        if not self.position:  # 若無持倉
            if self.momentum[0] > 0:
                self.buy()
                self.buy_price = self.data.close[0]  # 紀錄買入價格
        else:
            # 實施止損和止盈策略
            if self.data.close[0] <= self.buy_price * (1 - self.params.stop_loss):
                self.sell()  # 觸發止損
            elif self.data.close[0] >= self.buy_price * (1 + self.params.take_profit):
                self.sell()  # 觸發止盈

        self.daily_values.append(self.broker.getvalue())

    def stop(self):
        self.returns = pd.Series(self.daily_values).pct_change().dropna()
        annual_return = self.returns.mean() * 252
        annual_volatility = self.returns.std() * np.sqrt(252)
        downside_risk = self.returns[self.returns < 0].std() * np.sqrt(252)
        max_drawdown = (self.returns / self.returns.cummax() - 1).min()

        self.sharpe_ratio = (annual_return - 0.01) / annual_volatility
        self.sortino_ratio = (annual_return - 0.01) / downside_risk if downside_risk != 0 else None
        self.calmar_ratio = annual_return / abs(max_drawdown) if max_drawdown != 0 else None

        print(f"夏普比率: {self.sharpe_ratio:.2f}, Sortino比率: {self.sortino_ratio:.2f}, 卡瑟比率: {self.calmar_ratio:.2f}")

cerebro = bt.Cerebro()
data = yf.download('AAPL', start='2020-01-01', end='2021-01-01')
data = bt.feeds.PandasData(dataname=data)
cerebro.adddata(data)
print("\n")

# 使用 optstrategy 方法添加策略並指定參數範圍
cerebro.optstrategy(
    RiskAdjustedStrategy,
    stop_loss=[0.05, 0.1, 0.15],
    take_profit=[0.1, 0.2, 0.3],
    period=range(5, 21, 5)
)

# 設定初始資金和其他分析工具
cerebro.broker.setcash(100000.0)
cerebro.addsizer(bt.sizers.PercentSizer, percents=10)
cerebro.addanalyzer(btanalyzers.SharpeRatio, _name="sharpe", timeframe=bt.TimeFrame.Days, annualize=True)
cerebro.addanalyzer(btanalyzers.DrawDown, _name="drawdown")
cerebro.addanalyzer(btanalyzers.Returns, _name="returns")

# 執行策略優化
back = cerebro.run(maxcpus=1)


# 提取每個組合的回測結果
par_list = []
for run in back:
    for strategy in run:
        sharpe_ratio = strategy.analyzers.sharpe.get_analysis().get('sharperatio')
        returns = strategy.analyzers.returns.get_analysis().get('rnorm100')
        drawdown = strategy.analyzers.drawdown.get_analysis().get('max', {}).get('drawdown')

        if sharpe_ratio is not None:
            par_list.append([
                strategy.params.stop_loss,
                strategy.params.take_profit,
                strategy.params.period,
                returns,
                drawdown,
                sharpe_ratio
            ])

# 將結果轉換為 DataFrame 進行分析
par_df = pd.DataFrame(par_list, columns=['stop_loss', 'take_profit', 'period', 'return', 'drawdown', 'sharpe'])

# 找到夏普比率最高的參數組合
best_result = par_df.sort_values(by='sharpe', ascending=False).iloc[0]
print(f'最佳參數組合: 止損={best_result["stop_loss"]}, 止盈={best_result["take_profit"]}, 動量週期={best_result["period"]}')
print(f'對應的夏普比率: {best_result["sharpe"]:.2f}')


[*********************100%***********************]  1 of 1 completed




夏普比率: 1.08, Sortino比率: 1.45, 卡瑟比率: 0.02
夏普比率: 1.59, Sortino比率: 1.96, 卡瑟比率: 0.01
夏普比率: 1.76, Sortino比率: 2.33, 卡瑟比率: 0.03
夏普比率: 1.66, Sortino比率: 2.15, 卡瑟比率: 0.03
夏普比率: 1.53, Sortino比率: 2.16, 卡瑟比率: 0.02
夏普比率: 2.63, Sortino比率: 3.94, 卡瑟比率: 0.02
夏普比率: 2.35, Sortino比率: 3.37, 卡瑟比率: 0.03
夏普比率: 2.45, Sortino比率: 3.54, 卡瑟比率: 0.04
夏普比率: 0.93, Sortino比率: 1.27, 卡瑟比率: 0.02
夏普比率: 1.93, Sortino比率: 2.71, 卡瑟比率: 0.02
夏普比率: 1.63, Sortino比率: 2.22, 卡瑟比率: 0.03
夏普比率: 1.87, Sortino比率: 2.60, 卡瑟比率: 0.03
夏普比率: 1.31, Sortino比率: 1.77, 卡瑟比率: 0.02
夏普比率: 1.69, Sortino比率: 2.19, 卡瑟比率: 0.01
夏普比率: 1.41, Sortino比率: 1.86, 卡瑟比率: 0.02
夏普比率: 1.43, Sortino比率: 1.87, 卡瑟比率: 0.02
夏普比率: 1.07, Sortino比率: 1.47, 卡瑟比率: 0.02
夏普比率: 1.39, Sortino比率: 1.93, 卡瑟比率: 0.01
夏普比率: 1.71, Sortino比率: 2.35, 卡瑟比率: 0.02
夏普比率: 1.38, Sortino比率: 1.74, 卡瑟比率: 0.02
夏普比率: 1.17, Sortino比率: 1.64, 卡瑟比率: 0.02
夏普比率: 1.50, Sortino比率: 2.12, 卡瑟比率: 0.01
夏普比率: 1.39, Sortino比率: 1.88, 卡瑟比率: 0.02
夏普比率: 1.54, Sortino比率: 2.09, 卡瑟比率: 0.02
夏普比率: 1.48, Sortino比率: 2.03, 卡瑟比率: 0.0

[*********************100%***********************]  1 of 1 completed




夏普比率: 1.08, Sortino比率: 1.45, 卡瑟比率: 0.02
夏普比率: 1.59, Sortino比率: 1.96, 卡瑟比率: 0.01
夏普比率: 1.76, Sortino比率: 2.33, 卡瑟比率: 0.03
夏普比率: 1.66, Sortino比率: 2.15, 卡瑟比率: 0.03
夏普比率: 1.53, Sortino比率: 2.16, 卡瑟比率: 0.02
夏普比率: 2.63, Sortino比率: 3.94, 卡瑟比率: 0.02
夏普比率: 2.35, Sortino比率: 3.37, 卡瑟比率: 0.03
夏普比率: 2.45, Sortino比率: 3.54, 卡瑟比率: 0.04
夏普比率: 0.93, Sortino比率: 1.27, 卡瑟比率: 0.02
夏普比率: 1.93, Sortino比率: 2.71, 卡瑟比率: 0.02
夏普比率: 1.63, Sortino比率: 2.22, 卡瑟比率: 0.03
夏普比率: 1.87, Sortino比率: 2.60, 卡瑟比率: 0.03
夏普比率: 1.31, Sortino比率: 1.77, 卡瑟比率: 0.02
夏普比率: 1.69, Sortino比率: 2.19, 卡瑟比率: 0.01
夏普比率: 1.41, Sortino比率: 1.86, 卡瑟比率: 0.02
夏普比率: 1.43, Sortino比率: 1.87, 卡瑟比率: 0.02
夏普比率: 1.07, Sortino比率: 1.47, 卡瑟比率: 0.02
夏普比率: 1.39, Sortino比率: 1.93, 卡瑟比率: 0.01
夏普比率: 1.71, Sortino比率: 2.35, 卡瑟比率: 0.02
夏普比率: 1.38, Sortino比率: 1.74, 卡瑟比率: 0.02
夏普比率: 1.17, Sortino比率: 1.64, 卡瑟比率: 0.02
夏普比率: 1.50, Sortino比率: 2.12, 卡瑟比率: 0.01
夏普比率: 1.39, Sortino比率: 1.88, 卡瑟比率: 0.02
夏普比率: 1.54, Sortino比率: 2.09, 卡瑟比率: 0.02
夏普比率: 1.48, Sortino比率: 2.03, 卡瑟比率: 0.0

最佳參數組合: 止損=0.05, 止盈=0.2, 動量週期=10.0
對應的夏普比率: 2.56
